In [ ]:
# k1 corresponds to number of property ratios and k2 corresponds to number of shadow model for each property ratio
k1 = 7 
k2 = 6

In [ ]:
import os
import numpy as np
import torch
from tqdm import tqdm 
import xgboost as xgb
import math
from random import sample
from util import count_words
shadow_file_list = []
shadow_folder_path = ''
# List all files and directories in the folder
for file_name in os.listdir(shadow_folder_path):
    full_path = os.path.join(shadow_folder_path, file_name)
    if os.path.isfile(full_path):
        shadow_file_list.append(file_name)
        
shadow_file_list = sorted(shadow_file_list)
print(shadow_file_list)

shadow_ratio = np.array([[0.2 + i*0.1 for j in range(k2)] for i in range(k1)]).flatten()
print(shadow_ratio)


target_file_list = []
target_folder_path = ''
# List all files and directories in the folder
for file_name in os.listdir(target_folder_path):
    full_path = os.path.join(target_folder_path, file_name)
    if os.path.isfile(full_path):
        target_file_list.append(file_name)
        
target_file_list = sorted(target_file_list)
print(target_file_list)
target_ratio = [0.3, 0.3 ,0.3 ,0.5, 0.5, 0.5, 0.5, 0.7, 0.7, 0.7]
print(target_ratio)


In [ ]:
shadow_Dict_count = count_words(shadow_file_list, shadow_folder_path)
target_Dict_count = count_words(target_file_list, target_folder_path)

In [ ]:
# Calculate word frequency of each word appeared in the shadow model's generated text
# Select keyword using f_regression and train a meta_attack model

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import r_regression, f_regression, mutual_info_regression

#number of key_word used 
key_word_length = 35

X_train = []
name = []


# filtering out digits
for key in shadow_Dict_count.keys():
    # if np.min(Dict[key]) > filtering_min:
    # if np.min(Dict[key]) < 1000:
        if not key.isdigit():
            name.append(key)
            X_train.append(shadow_Dict_count[key])
    
X_train = np.array(X_train)
X_train = X_train.transpose()
print(X_train.shape)
y_train  = np.array(shadow_ratio)
print(y_train.shape)


# Select # keywords using f_regression
fs = SelectKBest(f_regression, k=key_word_length)
X_new = fs.fit_transform(X_train, y_train)
print(X_new.shape)

feature_name = fs.get_feature_names_out(name)

print(feature_name)


train_data = []
for feature in feature_name:
    train_data.append(np.array(shadow_Dict_count[feature]))
train_data = np.array(train_data)
train_data = np.transpose(train_data)
print(train_data.shape)


test_data = []
for feature in feature_name:
    if feature in target_Dict_count.keys():
        test_data.append(target_Dict_count[feature])
    else:
        test_data.append([0 for i in range(len(target_file_list))])
test_data = np.array(test_data)
test_data = np.transpose(test_data)
print(test_data.shape)


# Train a XG-Boost model
print("model = XG-Boost")
np.random.seed(0)
randperm = np.random.permutation(len(train_data))
n_train = int(math.ceil(len(train_data) * 0.75))
train_data, val_x = train_data[randperm[:n_train]], train_data[randperm[n_train:]]
train_y, val_y = y_train[randperm[:n_train]], y_train[randperm[n_train:]]

print(val_y)
dtrain = xgb.DMatrix(train_data, label=train_y)
dval = xgb.DMatrix(val_x, label=val_y)
evallist = [(dval, 'eval'), (dtrain, 'train')]
num_round = 100
bst = xgb.train({'objective': 'reg:squarederror'}, dtrain, num_round, evallist)
dtest = xgb.DMatrix(test_data, label=target_ratio)
test_pred = bst.predict(dtest)
# print(np.abs(test_pred - w_list))
print(np.abs(test_pred - target_ratio).mean())
print(test_pred)

    